# Prepare a Dataset: Working with XML Data using Python

how to get the text from a xml file
digital edition
XSLT can transform XML into other data formmats such as HTML, PDF, etc..
turn csv files into tei/xml formats
under what circumstances would this format be more effecient in managing data
sentence ID helps to identify where the sentences belonging to a specific topic from

In [1]:
# Python libraries for operating system interfaces and file names
import os, glob

# Pandas library
import pandas as pd

# Python standard library for XML processing
import xml.etree.ElementTree as ET

### Import Google Drive and set the datasets/parlamint directory

In [2]:
# import Google Drive
# this needs to be done every time you open a notebook that uses data from your Google Drive
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
# move to the correct folder where the dataset is located
%cd /content/drive/MyDrive/dse-ml-2026/materials/datasets/parlamint

/content/drive/MyDrive/dse-ml-2026/materials/datasets/parlamint


In [4]:
# check if you are in the correct directory
# you should now see the content of the materials/datasets/parlamint folder
%ls

ParlaMint-IT-en_2022-01-03-LEG18-Senato-sed-392.ana.xml
parlamint-it-is-2022.txt


### Read the sample TEI file from the current directory (datasets/parlamint) directory

In [5]:
# file name to the TEI sample file from a ParlaMint corpus
f_name = "ParlaMint-IT-en_2022-01-03-LEG18-Senato-sed-392.ana.xml"

In [6]:
# open the TEI file and parse it using the ET library
with open(f_name, "rb") as f:
    tree = ET.parse(f)

# set the document root of the XML document
root = tree.getroot()

### Getting all sentence elements (tei:s)

In [7]:
# namespace map with the namespaces we might use
NSMAP = {"tei": "http://www.tei-c.org/ns/1.0",
          "xml": "http://www.w3.org/XML/1998/namespace"
}

sentences = root.findall(".//tei:body//tei:s", namespaces=NSMAP)

#you could do the same using the lxml library
#lxml has additional functionality, for instance the xpath method for more complex queries
#sentences = tree.xpath("//tei:body//tei:s", namespaces=NSMAP)

sentences[:5]

[<Element '{http://www.tei-c.org/ns/1.0}s' at 0x7d9e05d6d710>,
 <Element '{http://www.tei-c.org/ns/1.0}s' at 0x7d9e05d6dc60>,
 <Element '{http://www.tei-c.org/ns/1.0}s' at 0x7d9e05d6e070>,
 <Element '{http://www.tei-c.org/ns/1.0}s' at 0x7d9e05d6ec50>,
 <Element '{http://www.tei-c.org/ns/1.0}s' at 0x7d9e05d6f010>]

### Getting the text from the first sentence in the XML file

![image.png](https://github.com/DHGraz/dse-ml-2026/blob/main/materials/2026-09-21_monday/04_bleier_scholger_prepare_a_dataset/img/parlamint.png?raw=1)

In [8]:
#with the method itertext() you can get the text of the sentences
#as you will see new line chracters \n are also included
#for more complex queries the use of the lxml library xpath method might be better suited
[t for t in sentences[0].itertext()]

['\n                     ',
 '\n',
 'The',
 '\n',
 'sitting',
 '\n',
 'was',
 '\n',
 'opened',
 '\n',
 'at',
 '\n',
 '11',
 '\n',
 'a.m',
 '\n',
 '.',
 '\n                  ']

In [9]:
#use the re library to apply a regular expression for filtering out the new line charcters
import re
[t for t in sentences[0].itertext() if not re.match("\n", t)]

['The', 'sitting', 'was', 'opened', 'at', '11', 'a.m', '.']

In [10]:
#we also want the sentence ID to reference sentences back
sentences[0].attrib["{http://www.w3.org/XML/1998/namespace}id"]

'ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.ana.seg1.1'

### Iterate over all sentences and the the sentence IDs and text of each sentence

In [11]:
# lists to store the texts and IDs
texts = []
ids = []

# loop over all sentence elements (tei:s)
for s in sentences:
    s_texts = [t for t in s.itertext() if not re.match("\n", t)]
    #join the list s_texts to a sentence string before adding it to the list texts
    #as you will see later the joining with whitespace produces whitespace at place we might not want it, e.g. before punctuation, further processing might be necessary
    texts.append(" ".join(s_texts))
    ids.append(s.attrib["{http://www.w3.org/XML/1998/namespace}id"])

### Read your texts and ids into a Pandas Dataframe

In [12]:
# create a Pandas DataFrame object and add the lists as data for the columns IDs and Texts
df = pd.DataFrame({"IDs":ids, "Texts":texts})

# show the first 5 rows of your DataFrame
df.head()

,IDs,Texts
0,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The sitting was opened at 11 a.m .
1,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The Minutes shall be read .
2,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,"Mr PUGLIA , Secretary , gave a reading of the ..."
3,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,PRESIDENT . - The debate is
4,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,"Since there were no comments , the Minutes wer..."


In [13]:
# write your DataFrame to a csv file, use tabs \t as separators
# the csv file will be saved to your Google Drive
df.to_csv("ParlaMint-IT-en_2022-01-03-LEG18-Senato-sed-392.ana.csv", sep="\t", index=False)

In [14]:
# now read in your new csv into a Pandas DataFrame
df = pd.read_csv("ParlaMint-IT-en_2022-01-03-LEG18-Senato-sed-392.ana.csv", sep="\t")

# show the first 5 rows of your DataFrame
df.head()

,IDs,Texts
0,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The sitting was opened at 11 a.m .
1,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The Minutes shall be read .
2,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,"Mr PUGLIA , Secretary , gave a reading of the ..."
3,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,PRESIDENT . - The debate is
4,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,"Since there were no comments , the Minutes wer..."


### Do some basic analysis of the corpus you have

In [15]:
# create an additional column in your DataFrame containing the length of each text
df["char_count"] = df["Texts"].str.len()
df.head()

,IDs,Texts,char_count
0,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The sitting was opened at 11 a.m .,34
1,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The Minutes shall be read .,27
2,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,"Mr PUGLIA , Secretary , gave a reading of the ...",90
3,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,PRESIDENT . - The debate is,27
4,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,"Since there were no comments , the Minutes wer...",58


In [16]:
# now we can do a basic analysis of this column to get an idea how long the texts in our corpus are
df.describe()

,char_count
count,50.000000
mean,156.640000
std,126.461378
min,13.000000
25%,46.750000
50%,144.000000
75%,201.500000
max,602.000000


In [17]:
# now let us inspect the very short texts
df[df["char_count"] < 50].head()

,IDs,Texts,char_count
0,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The sitting was opened at 11 a.m .,34
1,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The Minutes shall be read .,27
3,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,PRESIDENT . - The debate is,27
6,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The next item is :,18
12,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,I 'm asking to talk .,21


In [18]:
# if we do not need a column anymore, we can simply drop it
df.drop("char_count", axis=1, inplace=True)
df.head()

,IDs,Texts
0,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The sitting was opened at 11 a.m .
1,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,The Minutes shall be read .
2,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,"Mr PUGLIA , Secretary , gave a reading of the ..."
3,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,PRESIDENT . - The debate is
4,ParlaMint-IT_2022-01-03-LEG18-Senato-sed-392.a...,"Since there were no comments , the Minutes wer..."
